# 03 · Business Insights & Recommendations

This notebook translates model outputs into a narrative that an HR leader can act on.
The goal is not to present statistics — it is to answer the business question:

> **Which employees are most likely to resign in the next 90 days, and what should we do about it?**

---

## Key Findings Summary

| Driver | Direction | Business Implication |
|--------|-----------|----------------------|
| Compa Ratio | Employees below market (compa < 0.90) attrite at 2.3× the rate | Run a compensation review; close gaps for flight-risk employees |
| Engagement Score | Each 1-point drop in engagement raises attrition probability by ~8pp | Pulse survey every quarter; flag teams with declining scores |
| Overtime | Employees averaging >12 hrs overtime/week show 1.8× attrition rate | Monitor burnout by department; enforce WLB policies in Sales |
| Tenure | Spike in first 12 months, then stabilizes until 20+ years | Invest in structured onboarding; track 90-day engagement |
| Stagnation | >3 years since last promotion correlates with elevated risk | Tie talent reviews to internal mobility paths |

---

## Recommended CHRO Actions
1. **Compensation audit** — flag all employees with compa_ratio < 0.90 who are rated 4+ performers
2. **Engagement intervention** — identify the 3 lowest-engagement teams and schedule skip-level conversations
3. **Manager accountability** — share department-level attrition rates with business leaders quarterly
4. **Onboarding investment** — create a structured 90-day experience for new hires in Sales and Engineering


In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/employee_data.csv')

with open('../models/rf_attrition_model.pkl','rb') as f:
    saved = pickle.load(f)

model    = saved['model']
le       = saved['encoders']
FEATURES = saved['features']

In [ ]:
# --- SCORE ALL EMPLOYEES ---
df_enc = df.copy()
for c in ['gender','department']:
    df_enc[c] = le[c].transform(df[c])

df['attrition_probability'] = model.predict_proba(df_enc[FEATURES])[:,1]
df['risk_tier'] = pd.cut(df['attrition_probability'],
                          bins=[0,0.25,0.50,1.0],
                          labels=['Low','Medium','High'])

print(df['risk_tier'].value_counts())
print(f"\nHigh-risk employees: {(df['risk_tier']=='High').sum()}")

In [ ]:
# --- HIGH-RISK FLIGHT RISK TABLE ---
# Simulate the output HR would actually use
flight_risk = (df[df['risk_tier']=='High']
               [['employee_id','department','job_level','tenure_years',
                 'compa_ratio','engagement_score','attrition_probability']]
               .sort_values('attrition_probability', ascending=False)
               .head(20))

flight_risk['attrition_probability'] = flight_risk['attrition_probability'].map('{:.0%}'.format)
flight_risk['compa_ratio'] = flight_risk['compa_ratio'].map('{:.3f}'.format)
print("Top 20 highest-risk employees:")
print(flight_risk.to_string(index=False))

In [ ]:
# --- RETENTION COST ESTIMATE ---
# Rule-of-thumb: replacing an employee costs 50–200% of annual salary
avg_salary        = df['salary'].mean()
high_risk_count   = (df['risk_tier']=='High').sum()
replacement_low   = high_risk_count * avg_salary * 0.5
replacement_high  = high_risk_count * avg_salary * 1.5
expected_attrited = (df['attrition_probability'] * df['salary']).sum()

print(f"Employees in 'High' risk tier:      {high_risk_count}")
print(f"Average salary:                     ${avg_salary:,.0f}")
print(f"Est. replacement cost (low):        ${replacement_low:,.0f}")
print(f"Est. replacement cost (high):       ${replacement_high:,.0f}")
print(f"\n→ Reducing high-risk attrition by 20% saves ${replacement_low*0.20:,.0f}–${replacement_high*0.20:,.0f}")

In [ ]:
# --- FIGURE: Attrition Risk Distribution by Department ---
dept_risk = (df.groupby('department')['attrition_probability']
               .agg(['mean','count'])
               .rename(columns={'mean':'avg_risk','count':'n'})
               .sort_values('avg_risk', ascending=True))

fig, ax = plt.subplots(figsize=(8,4.5))
colors = ['#FF7B72' if v >= 0.12 else '#E3B341' if v >= 0.08 else '#3FB950'
          for v in dept_risk['avg_risk']]
dept_risk['avg_risk'].mul(100).plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Average Predicted Attrition Probability (%)')
ax.set_title('Predicted Attrition Risk by Department')
plt.tight_layout()
plt.savefig('../outputs/figures/fig5_risk_by_department.png', dpi=150, bbox_inches='tight')
plt.show()

## Limitations & Future Work

- **Sample size:** 1,470 synthetic employees — real enterprise deployments need 500+ attrition events for stable models
- **Causal inference:** Feature importance ≠ causation; interpret as correlates, not levers
- **Model drift:** Retrain quarterly as workforce composition shifts
- **Fairness audit:** Check that the model does not encode demographic proxies — run a disparate impact analysis before deployment
- **Next step:** Replace synthetic data with Workday RaaS extract → retrain → deploy as a Tableau dashboard refreshed monthly

---
*GreedyAlgo Analytics · Hari Vemula · github.com/harivemula/pa-attrition-predictor*
